In [12]:
# --- Импорты ---
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader

# --- Устройство ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# --- Данные ---
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# --- Обработка данных ---
# Выделяем признаки и целевую переменную
X = train_df.drop(columns=['target']).values
y = train_df['target'].values

X_test = test_df.values

# Масштабируем признаки
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_test = scaler.transform(X_test)

# Делим тренировочную выборку на обучающую и валидационную
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.05, random_state=42)

# --- Создаем Dataset ---
class CustomDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long) if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        else:
            return self.X[idx]

train_dataset = CustomDataset(X_train, y_train)
val_dataset = CustomDataset(X_val, y_val)
test_dataset = CustomDataset(X_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# --- Построение модели ---
class FeatureBlock(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, out_features)
        self.act = nn.GELU()
        self.drop1 = nn.Dropout(0.01)
        self.fc2 = nn.Linear(out_features, out_features)
        self.drop2 = nn.Dropout(0.01)
        self.shortcut = nn.Linear(in_features, out_features)

    def forward(self, x):
        identity = self.shortcut(x)
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop1(x)
        x = self.fc2(x)
        x = self.drop2(x)
        return x + identity

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            FeatureBlock(16, 64),    # 16 -> 64
            nn.BatchNorm1d(64),
            nn.Dropout(0.01),
            FeatureBlock(64, 256),   # 64 -> 256
            nn.BatchNorm1d(256),
            nn.Dropout(0.01),
            FeatureBlock(256, 128),  # 256 -> 128
            nn.BatchNorm1d(128),
            nn.Dropout(0.01)
        )
        self.fc_out = nn.Linear(128, 2)

    def forward(self, x):
        x = self.feature_extractor(x)
        x = self.fc_out(x)
        return x

model = Net().to(device)

# --- Определение функции потерь и оптимизатора ---
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# --- Обучение модели ---
n_epochs = 30

for epoch in range(n_epochs):
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Валидация
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            val_loss += loss.item()

            preds = outputs.argmax(1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total

    print(f"Epoch {epoch+1}/{n_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

# --- Предсказания на тесте ---
model.eval()
test_preds = []

with torch.no_grad():
    for X_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        preds = outputs.argmax(1)
        test_preds.extend(preds.cpu().numpy())

# --- Сохраняем предсказания ---
submission = pd.DataFrame({'target': test_preds})
submission.to_csv('answers.csv', index=False, header=False)
print('Сохранено в answers.csv')

Using device: cpu
Epoch 1/30 | Train Loss: 0.3740 | Val Loss: 0.2523 | Val Acc: 0.8769
Epoch 2/30 | Train Loss: 0.3629 | Val Loss: 0.2212 | Val Acc: 0.9077
Epoch 3/30 | Train Loss: 0.3660 | Val Loss: 0.2396 | Val Acc: 0.8923
Epoch 4/30 | Train Loss: 0.3457 | Val Loss: 0.2170 | Val Acc: 0.9231
Epoch 5/30 | Train Loss: 0.3517 | Val Loss: 0.2707 | Val Acc: 0.9077
Epoch 6/30 | Train Loss: 0.3230 | Val Loss: 0.1982 | Val Acc: 0.8923
Epoch 7/30 | Train Loss: 0.3424 | Val Loss: 0.1941 | Val Acc: 0.9077
Epoch 8/30 | Train Loss: 0.3154 | Val Loss: 0.2125 | Val Acc: 0.9077
Epoch 9/30 | Train Loss: 0.3167 | Val Loss: 0.1985 | Val Acc: 0.9077
Epoch 10/30 | Train Loss: 0.3376 | Val Loss: 0.2102 | Val Acc: 0.8923
Epoch 11/30 | Train Loss: 0.3148 | Val Loss: 0.2138 | Val Acc: 0.8923
Epoch 12/30 | Train Loss: 0.3286 | Val Loss: 0.1731 | Val Acc: 0.9077
Epoch 13/30 | Train Loss: 0.3299 | Val Loss: 0.2003 | Val Acc: 0.9231
Epoch 14/30 | Train Loss: 0.3178 | Val Loss: 0.2067 | Val Acc: 0.8769
Epoch 15/30